**DATA CLEANING NOTES PER DATASET**

1. Gemini Dataset
    - A bit easy to clean if we were to base the cleaning on the

2. Claude Dataset
    - Needs a bit of cleaning since it has a lot of curly braces
    - It has equations and code 
    - to clean: remaining greek operators, fractions, and code

3. MGTBench Dataset (ChatGPT)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

4. MGTBench Dataset (Human-written)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

5. BAWE Corpus Dataset
    - Needs the most cleaning since it an HTML file 
    - Need to convert it into an HTML file so we can clean it properly

In [67]:
# Import the necessary libraries for cleaning the data
import os
import re
import pandas as pd
import numpy as numpy
from pathlib import Path
from tqdm import tqdm
import ftfy
import ast

pd.set_option('display.max_colwidth', 150)
tqdm.pandas(desc="Cleaning Text")

print("Libraries has been imported!")

Libraries has been imported!


In [68]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")

Data Cleaning Paths Ready:
  Reading AI Raw Data:        C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\ai
  Reading Human Raw Data:     C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\human
  Saving AI Processed Data:   C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai
  Saving Human Processed Data:C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human


**METHODS FOR CLEANING DATA**

- Cleans math texts
- Cleans texts that has citations
- Cleans texts that has code in it
- Strips reference lists
- Cleans texts that has any numberings in it
- Checks whether a text is creative works and it will remove that since creatives is not considered as an academic text

In [69]:
def clean_math_texts(text):
    """
    Cleans math equations, LaTeX expressions, and Unicode mathematical symbols from text
    by replacing them with [[EQUATION]] placeholders.
    """
    if not isinstance(text, str):
        return text

    # Normalize literal escaped newlines and citation markers
    text = text.replace('\\n', ' ')
    text = re.sub(r'\[\d+\]', '', text)

    # LaTeX environments
    text = re.sub(r'\\begin\{[a-zA-Z0-9\*]+\}.*?\\end\{[a-zA-Z0-9\*]+\}', ' [[EQUATION]] ', text, flags=re.DOTALL)

    # LaTeX block and display math
    text = re.sub(r'\$\$.*?\$\$', ' [[EQUATION]] ', text, flags=re.DOTALL)
    text = re.sub(r'\\\[.*?\\\]', ' [[EQUATION]] ', text, flags=re.DOTALL)

    # LaTeX inline math
    text = re.sub(r'\$([^\$\n]+)\$', ' [[EQUATION]] ', text)
    text = re.sub(r'\\\((.*?)\\\)', ' [[EQUATION]] ', text)

    # Common LaTeX commands
    text = re.sub(r'\\[a-zA-Z]+(\{.*?\})*', ' [[EQUATION]] ', text)

    # Integrals with bounds and differentials
    text = re.sub(r'[∮∫][₀-₉⁰-⁹\^]*[^\.\n]*?d[A-Za-z]+', ' [[EQUATION]] ', text)

    # Algebraic equations containing '=' or comparison operators
    text = re.sub(r'\b[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^]+\s*(?:<=|>=|!=|==|=|<|>|≠|≤|≥|≈)\s*[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^\s\._]+', ' [[EQUATION]] ', text)

    # Exponents and derivatives
    text = re.sub(r'\b[a-zA-Z0-9\(\)]+\^[a-zA-Z0-9\(\)\+\-]+\b', ' [[EQUATION]] ', text)
    text = re.sub(r'\bd[A-Za-z]/d[A-Za-z]\b', ' [[EQUATION]] ', text)

    # "n choose k" style combinatorics notation
    text = re.sub(r'\([a-zA-Z0-9\s\+\-]+choose[a-zA-Z0-9\s\+\-]+\)', '[[EQUATION]]', text)

    # Catches sentences that survived token-level cleaning but are still mostly fragments/placeholders
    cleaned_sentences = []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    prev_was_equation = False

    for sent in sentences:
        placeholder_count = sent.count('[[EQUATION]]')
        non_placeholder_text = re.sub(r'\[\[EQUATION\]\]', '', sent)
        word_count = len(re.findall(r'[a-zA-Z]{3,}', non_placeholder_text))

        # If a sentence has 2+ placeholders and very few real words around them,
        # it's fragment soup, so we need to collapse to a single [[EQUATION]]
        is_fragment_heavy = placeholder_count >= 2 and word_count < 6

        if is_fragment_heavy or (placeholder_count >= 1 and word_count == 0):
            if not prev_was_equation:
                cleaned_sentences.append("[[EQUATION]]")
            prev_was_equation = True
        else:
            cleaned_sentences.append(sent)
            prev_was_equation = False

    text = ' '.join(cleaned_sentences)
    text = re.sub(r'(\[\[EQUATION\]\]\s*){2,}', '[[EQUATION]]', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def strip_reference_list(text):
    """Cuts off everything from the first 'Sources:' / 'References' """

    match = re.search(r'\n?(Sources|References|Bibliography):', text, flags=re.IGNORECASE)

    if match:
        return text[:match.start()].strip()

    return text


def clean_code_texts(text):
    """
        Cleans code blocks and inline code snippets from text by replacing them 
        with [[CODE]] placeholders.
    """

    if not isinstance(text, str):
        return text

    # Replace markdown fenced code blocks
    text = re.sub(r'```[a-zA-Z0-9_\+\#-]*\n?[\s\S]*?```', ' [[CODE]] ', text)

    # Replace inline code snippets
    text = re.sub(r'`[^`\n]+`', ' [[CODE]] ', text)

    # Consolidate consecutive [[CODE]] tags and normalize whitespace
    text = re.sub(r'(\[\[CODE\]\]\s*){2,}', '[[CODE]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_citations(text):
    """
        Replaces any citations whether intext to [[CITATION]]
    """
    if not isinstance(text, str):
        return text

    # Bracketed numeric citations: [1], [12], [1,2]
    text = re.sub(r'\[\d+(,\s*\d+)*\]', ' [[CITATION]] ', text)

    # Parenthetical citations: (Smith, 2020), (Smith et al., 2020), (Smith & Jones, 2020),
    # (Smith, 2020; Lee, 2019), (Smith 2020) — comma optional, semicolon-joined multiples
    text = re.sub(
        r'\([A-Z][a-zA-Z\.\s,&]*?(?:et al\.)?\s*,?\s*\d{4}[a-z]?(?:\s*;\s*[A-Z][a-zA-Z\.\s,&]*?\d{4}[a-z]?)*\)',
        ' [[CITATION]] ', text
    )

    # Narrative citations: "Smith (2020)", "Smith et al. (2020)"
    text = re.sub(r'\b[A-Z][a-zA-Z]+(?:\set al\.)?\s\(\d{4}[a-z]?\)', ' [[CITATION]] ', text)

    text = re.sub(r'(\[\[CITATION\]\]\s*){2,}', '[[CITATION]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_complexity_notation(text):
    """
        Cleans the rows that has any Big-O notation in the dataset
    """

    if not isinstance(text, str):
        return text

    # Big-O, Big-Theta, Big-Omega notation: O(n), O(log n), O(n^2), Θ(n), Ω(n log n)
    # Requires the letter to be standalone (not preceded by another letter) to avoid
    # false matches like "info(x)" or "to(n)" — also drops lowercase 'o' since it's
    # too ambiguous with common English words even with the lookbehind guard.
    text = re.sub(r'(?<![a-zA-Z])[OΘΩ]\(\s*[a-zA-Z0-9\s\^\+\-\*/,]*\)', ' [[COMPLEXITY]] ', text)

    text = re.sub(r'(\[\[COMPLEXITY\]\]\s*){2,}', '[[COMPLEXITY]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_list_numbering(text):
    """
        Cleans texts that has any numbering in it
        e.g. 1. 2. ...
    """

    if not isinstance(text, str):
        return text

    # Numbered list markers: "1.", "2.", "\n3."
    text = re.sub(r'(?:^|(?<=[\s:;\.]))\d{1,2}\.\s+(?=[A-Za-z])', ' ', text)

    # Letter list markers
    text = re.sub(r'(?:^|(?<=[\s:;\.]))[a-zA-Z][\.\)]\s+(?=[A-Za-z0-9])', ' ', text)

    # Roman Numerals list markers
    text = re.sub(r'(?:^|(?<=[\s:;\.]))(?:i{1,3}|iv|v|vi{0,3}|ix|x)[\.\)]\s+', ' ', text, flags=re.IGNORECASE)

    # Dash/bullet markers: "- Puts more money...", "\n- Allows workers..."
    text = re.sub(r'(?:^|\n)\s*-\s+', ' ', text)

    # Cleans up any whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def is_academic_content(prompt="", text=""):
    """
        This function returns False if either the prompt of the text contains a creative,
        fictional, commercial script, or stage-direction markers that fall outside the
        academic scope
    """

    combined = (str(prompt) + " " + str(text)).lower()

    creative_keywords = [
        'script', 'commercial', 'advertisement', 'ad script', 'screenplay', 
        'short story', 'story', 'novel', 'fiction', 'fairy tale', 'fairytale', 
        'poem', 'poetry', 'lyrics', 'song', 'monologue', 'dialogue between', 
        'playwright', 'haiku', 'sonnet', 'screenwriter', 'broadway', 'fanfiction'
    ]

    if any(kw in combined for kw in creative_keywords):
        return False

    # Stage Direction & Script Structural Markers in response texts
    script_patterns = [
        r'\[visual:', r'\[audio:', r'\[music', r'\[sfx:', r'\[scene', 
        r'\[camera', r'\[upbeat', r'\[fade', r'\bnarrator:', r'\bint\.\s', r'\bext\.\s'
    ]

    if any(re.search(pattern, combined) for pattern in script_patterns):
        return False

    return True

def clean_urls(text):
    """
        Replaces URLS (even with or without a hyperlink label) with a 
        [[URL]] placeholder
    """

    if not isinstance(text, str):
        return text

    text = re.sub(r'https?://\S+', ' [[URL]] ', text)

    # www.-prefixed URLs without a scheme
    text = re.sub(r'\bwww\.\S+', ' [[URL]] ', text)

    text = re.sub(r'(\[\[URL\]\]\s*){2,}', '[[URL]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

Since most datasets have a lot of placeholders which can render the dataset unsable/useful for both spaCy and ELECTRA. 

In [70]:
def placeholder_density(text):
    """
        This method is meant to remove the rows that have too many
        tokens on the cleaned data
    """

    if not isinstance(text, str):
        return 0

    placeholder_count = len(re.findall(r'\[\[EQUATION\]\]|\[\[CODE\]\]|\[\[CITATION\]\]|\[\[COMPLEXITY\]\]|\[\[URL\]\]', text))
    word_count = len(re.findall(r'\b[a-zA-Z]{2,}\b', text))
    total = placeholder_count + word_count
    
    return placeholder_count / total if total > 0 else 0

def clean_pipeline(text):
    """
        Combined cleaning pipeline for all of the datasets
    """

    text = clean_code_texts(text)
    text = clean_urls(text)
    text = clean_complexity_notation(text)
    text = strip_reference_list(text)
    text = clean_citations(text)
    text = clean_list_numbering(text)
    text = clean_math_texts(text)

    return text

In [71]:
import IPython.display as ipd

def extract_claude_prompt_and_response(text):
    """
        Extracts human prompt and claude/gpt response specifically from 
        Claude dataset's dictionary-style conversation strings.
        Handles plain text datasets by returning an empty prompt and raw text.
    """
    if not isinstance(text, str):
        return "", str(text)

    if text.strip().startswith('[') and "'from'" in text and "'value'" in text:
        try:
            data = ast.literal_eval(text)
            prompt = ""
            raw_response = ""

            for turn in data:
                if isinstance(turn, dict):
                    role = turn.get('from')
                    val = turn.get('value', '')

                    if role == 'human' and not prompt:
                        prompt = val
                    elif role in ('gpt', 'assistant'):
                        raw_response += val + " "

            return prompt.strip(), raw_response.strip()

        # Fallback regex incase it encountered a problem in parsing
        except Exception:
            prompt_match = re.search(r"\{'from':\s*'human',\s*'value':\s*\"\"?(.*?)\"\"?\}", text, flags=re.DOTALL)
            resp_match = re.search(r"\{'from':\s*'(?:gpt|assistant)',\s*'value':\s*\"\"?(.*?)\"\"?\}", text, flags=re.DOTALL)
            prompt = prompt_match.group(1) if prompt_match else ""
            raw_response = resp_match.group(1) if resp_match else text

            return prompt.strip(), raw_response.strip()

    return "", text.strip()


def clean_claude_dataset(claude_csv_path, sample_size=200, density_threshold=0.4):
    """
        Cleans the Claude AI dataset, filters out non-academic creative prompts,
        extracts prompt and cleaned response, filters out placeholder-dense rows,
        counts tag insertions, displays summary & sample tables, and saves output
        to PROCESSED_AI_DIR.
    """
    if not claude_csv_path.exists():
        print(f"File not found at: {claude_csv_path}")
        return None

    print(f"Loading {'first ' + str(sample_size) if sample_size else 'all'} rows from {claude_csv_path.name}...")
    df_raw = pd.read_csv(claude_csv_path, nrows=sample_size)
    prompts = []
    cleaned_responses = []
    dropped_creative = 0
    dropped_density = 0

    print("Cleaning & filtering Claude dataset...")
    for raw_text in tqdm(df_raw['conversations'], desc="Processing Rows"):
        prompt, raw_response = extract_claude_prompt_and_response(str(raw_text))

        # Filter out non-academic creative writing prompts (stories, scripts, poems)
        if not is_academic_content(prompt, raw_response):
            dropped_creative += 1
            continue

        cleaned_resp = clean_pipeline(raw_response)

        # Filter out rows that are mostly placeholder tokens after cleaning
        # (too notation-dense to yield meaningful stylometric features)
        density = placeholder_density(cleaned_resp)
        if density >= density_threshold:
            dropped_density += 1
            continue

        prompts.append(prompt)
        cleaned_responses.append(cleaned_resp)

    # Build clean output DataFrame with prompt and cleaned_text columns
    df_processed = pd.DataFrame({
        'prompt': prompts,
        'cleaned_text': cleaned_responses
    })

    summary_data = {
        "Metric": [
            "Total Rows Loaded",
            "Dropped (Non-Academic/Creative)",
            "Dropped (Too Placeholder-Dense)",
            "Total Academic Rows Kept",
            "[[EQUATION]] Tags Inserted",
            "[[CODE]] Tags Inserted",
            "[[CITATION]] Tags Inserted",
            "[[COMPLEXITY]] Tags Inserted",
            "[[URL]] Tags Inserted"
        ],
        "Count": [
            len(df_raw),
            dropped_creative,
            dropped_density,
            len(df_processed),
            df_processed['cleaned_text'].str.count(r'\[\[EQUATION\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[CODE\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[CITATION\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[COMPLEXITY\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[URL\]\]').sum()
        ]
    }
    df_summary = pd.DataFrame(summary_data)

    print("\n--- CLEANING SUMMARY ---")
    ipd.display(df_summary)

    # Save processed data to PROCESSED_AI_DIR
    filename = f"claude_dataset_cleaned_{sample_size}.csv" if sample_size else "claude_dataset_cleaned.csv"
    output_path = PROCESSED_AI_DIR / filename
    df_processed.to_csv(output_path, index=False)
    print(f"\nSuccessfully saved cleaned dataset ({len(df_processed)} rows) to:\n  {output_path.resolve()}")

    # Display first 20 rows table (prompt & cleaned_text)
    print("\n--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---")
    ipd.display(df_processed.head(20))

    return df_processed


claude_dataset = RAW_AI_DIR / 'claude_dataset.csv'
df_cleaned_claude = clean_claude_dataset(claude_dataset, sample_size=200)

Loading first 200 rows from claude_dataset.csv...
Cleaning & filtering Claude dataset...


Processing Rows: 100%|██████████| 200/200 [00:00<00:00, 637.03it/s]


--- CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Loaded,200
1,Dropped (Non-Academic/Creative),118
2,Dropped (Too Placeholder-Dense),0
3,Total Academic Rows Kept,82
4,[[EQUATION]] Tags Inserted,309
5,[[CODE]] Tags Inserted,117
6,[[CITATION]] Tags Inserted,58
7,[[COMPLEXITY]] Tags Inserted,30
8,[[URL]] Tags Inserted,12



Successfully saved cleaned dataset (82 rows) to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\claude_dataset_cleaned_200.csv

--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---


,prompt,cleaned_text
0,"Generate a step-by-step proof for the binomial theorem, explaining each step in detail using multiple examples to illustrate the concepts involved...",The binomial theorem states that for any real numbers x and y and a non-negative integer n: (x + [[EQUATION]] ∑ [[EQUATION]] −k) [[EQUATION]] Wher...
1,Create a function in Python code that accepts a string input and outputs the number of vowels in the string. The function signature and a sample c...,"Here's a Python function that counts the number of vowels in a given string: [[CODE]] This function does the following: We define a set of vowels,..."
2,"Write a well-articulated series of paragraphs explaining how to effectively structure one's writing through the use of topic sentences, transition...","To effectively structure one's writing, it's important to employ clear topic sentences, smooth transitions, a consistent theme, and proper punctua..."
3,"Develop a general overview of algorithms and data structures commonly used in software engineering, covering topics like searching, sorting, recur...",Here's a comprehensive overview of algorithms and data structures commonly used in software engineering: Searching Algorithms Linear Search - Time...
4,Derive a proof for the Pythagorean Theorem relating the sides of a right triangle and use it to help students understand and remember the key math...,Certainly! I'll provide a proof for the Pythagorean Theorem and then discuss how to help students understand and remember this important mathemati...
5,•Faraday's experiments on induction which established the basic principles. Discuss how Faraday's observations lead to the concept of magnetic fl...,"Faraday's Experiments and Magnetic Flux: In the 1830s, Michael Faraday conducted a series of experiments that established the fundamental principl..."
6,Imagine that you have two claims that appear to conflict with one another. Analyze the reasons and evidence provided for both claims. Look for way...,Thank you for this interesting thought exercise. I'll do my best to analyze and reconcile two hypothetical conflicting claims in a reasoned way. S...
7,Explain how to solve a system of three nonlinear equations in three unknowns using Newton's method. Provide detailed step-by-step instructions alo...,Newton's method can be used to solve systems of nonlinear equations by finding successive approximations to the solution. For a system of 3 equati...
8,Summarize the U.S court case Miranda v. Arizona (1966) in which the Supreme Court established that the police must inform individuals of their Mir...,Miranda [[CITATION]] was a landmark U. Supreme Court case that established important constitutional protections for criminal suspects during polic...
9,"Write a short Python program that generates 10 random integer numbers between 1 and 100, calculates the sum and average of those numbers, and pri...",Here's a short Python program that does what you've described: [[CODE]] This program does the following: It imports the [[CODE]] module to generat...


In [72]:
def clean_claude_dataset(dataset):
    
    pass

In [73]:
def clean_mgtbench_human_dataset(dataset):

    pass

In [74]:
def clean_mgtbench_ai_dataset(dataset):

    pass

In [75]:
def clean_bawe_corpus_dataset(dataset):

    pass